# 04 – Streaming (Wikipedia changes through Kafka)

A Spark Structured Streaming query reads the live Wikipedia change events
from Kafka and keeps the changes made to articles related to the IMDb
dataset. This notebook explains the pipeline and shows the results of the
25-minute test window.

**Prerequisites**: the enrichment step (notebook 03), then, inside the
Spark container:

```bash
docker compose exec spark python jobs/streaming.py lookup
docker compose exec spark python jobs/streaming.py run --fresh --minutes 27 --starting-offsets latest
docker compose run --rm producer --minutes 25          # in a second terminal
```

Code: `scripts/wiki_producer.py` (producer), `src/imdb_project/streaming.py`
and `jobs/streaming.py` (Spark).

In [1]:
import pandas as pd
from pyspark.sql import functions as F

from imdb_project.spark import get_spark

spark = get_spark("notebook-streaming")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from imdb_project import streaming as st
from imdb_project.config import STREAMING_DIR

lookup = spark.read.parquet(str(STREAMING_DIR / "lookup"))
matches = spark.read.parquet(str(STREAMING_DIR / "matches"))
streamed = spark.read.parquet(str(STREAMING_DIR / "windows"))
final = spark.read.parquet(str(STREAMING_DIR / "windows_final"))
progress = spark.read.json(str(STREAMING_DIR / "progress.jsonl"))

## 1. Architecture

```
Wikimedia EventStreams  --SSE-->  producer (Python)  -->  Kafka topic       -->  Spark Structured Streaming
(recentchange, live)              keeps English           wikipedia-changes       join with the lookup
                                  Wikipedia                                       -> matches (Parquet)
                                                                                  -> 5-min windows (Parquet)
                                                                                  -> metrics (JSON lines)
```

- The **producer** reads the public `recentchange` stream (Server-Sent
  Events), keeps the English Wikipedia (`enwiki`) changes and sends each
  event as JSON to Kafka. It reconnects by itself when the connection drops.
- **Kafka** decouples the two sides: the producer and Spark can run at
  different speeds, and Spark can restart from its checkpoint.
- **Spark** reads the topic, parses the JSON with an explicit schema and
  joins the events with the lookup table.

## 2. Relevance filter

A change is relevant when it is made to the **exact Wikipedia article** of
an enriched film or principal. These articles were resolved from IMDb ids
through Wikidata during the enrichment, so the filter has no homonym: a
change to a page called "Wildwood" only matches if it is the article of the
film *Wildwood* in our dataset. Page titles are compared case-insensitively.

A broader filter (titles and names of all popular IMDb films, or film
categories) was tested first: it gave about 130 matches in 25 minutes but
also homonyms. The exact filter gives fewer matches, enough for the test
window, all of them certain.

In [2]:
print(f"{lookup.count():,} articles in the lookup")
lookup.groupBy("entity_type").count().toPandas()

13,913 articles in the lookup


,entity_type,count
0,person,8116
1,movie,5797


## 3. Late data

Each event has two times: `event_time`, when the change was made on
Wikipedia, and `kafka_time`, when it reached Kafka. The windows are built on
**event time**, so that a late event is counted in the window where it
happened.

- **Watermark of 5 minutes** (`withWatermark("event_time", "5 minutes")`):
  Spark waits for events up to 5 minutes later than the latest event time
  seen. An older event is not added to the window counts any more, and a
  window is final (written) once the watermark has passed its end.
- The stream of Wikipedia changes arrives a few seconds late at most (see
  the delays below), so 5 minutes leaves a large margin while writing the
  windows soon.
- **Output mode `append`**: each window is written once, when it is final.
  This is the only mode supported by the Parquet sink for aggregations.
- The relevant changes themselves are all kept, whatever their delay (the
  `matches` output has no aggregation).
- **End of the test**: the watermark only moves when newer events arrive,
  so the last windows are never final while the stream runs. When the query
  stops, the window counts are computed again in batch from all the saved
  matches (`windows_final`).

In [3]:
matches.groupBy("entity_type").agg(
    F.count(F.lit(1)).alias("changes"),
    F.countDistinct("title").alias("pages"),
    F.sum(F.col("bot").cast("int")).alias("bot_changes"),
    F.round(F.avg("ingestion_delay_s"), 1).alias("avg_delay_s"),
    F.max("ingestion_delay_s").alias("max_delay_s"),
).toPandas()

,entity_type,changes,pages,bot_changes,avg_delay_s,max_delay_s
0,movie,7,5,0,3.9,12
1,person,4,3,0,6.0,10


## 4. Results of the test window

In [4]:
matches.orderBy("event_time").select(
    "event_time", "entity_type", "imdb_name", "title", "user", "bot", "size_change",
    "ingestion_delay_s").toPandas()

,event_time,entity_type,imdb_name,title,user,bot,size_change,ingestion_delay_s
0,2026-11-14 09:01:30,person,Russell Brand,Russell Brand,GreyZoneLife,False,748.0,10
1,2026-11-14 09:04:05,movie,Eyes Wide Shut,Eyes Wide Shut,~2026-48906-71,False,0.0,12
2,2026-11-14 09:04:16,movie,Eyes Wide Shut,Eyes Wide Shut,~2026-48906-71,False,NaN,0
3,2026-11-14 09:06:10,person,Russell Brand,Russell Brand,GreyZoneLife,False,47.0,6
4,2026-11-14 09:13:18,movie,Surrounded,Surrounded (2023 film),~2026-53138-06,False,4.0,1
5,2026-11-14 09:13:36,movie,Surrounded,Surrounded (2023 film),~2026-53138-06,False,8.0,2
6,2026-11-14 09:13:44,movie,True Romance,True Romance,MovieArchivist313,False,7.0,3
7,2026-11-14 09:14:08,person,Letitia Wright,Letitia Wright,~2026-53138-06,False,5.0,6
8,2026-11-14 09:17:51,movie,Ghost,Ghost (1990 film),~2026-60324-58,False,-2.0,7
9,2026-11-14 09:17:54,movie,Jana Gana Mana,Jana Gana Mana (2022 film),TamilSelvan,False,-12.0,2


In [5]:
keys = ["window_start", "window_end", "entity_type"]
(final.select(*keys, F.col("changes").alias("changes_batch_at_end"))
 .join(streamed.select(*keys, F.col("changes").alias("changes_written_by_stream")), keys, "left")
 .orderBy(*keys).toPandas())

,window_start,window_end,entity_type,changes_batch_at_end,changes_written_by_stream
0,2026-11-14 09:00:00,2026-11-14 09:05:00,movie,2,2.0
1,2026-11-14 09:00:00,2026-11-14 09:05:00,person,1,1.0
2,2026-11-14 09:05:00,2026-11-14 09:10:00,person,1,1.0
3,2026-11-14 09:10:00,2026-11-14 09:15:00,movie,3,NaN
4,2026-11-14 09:10:00,2026-11-14 09:15:00,person,1,NaN
5,2026-11-14 09:15:00,2026-11-14 09:20:00,movie,2,NaN
6,2026-11-14 09:15:00,2026-11-14 09:20:00,person,1,NaN


The windows written by the stream have the same counts as the batch counts:
no event arrived later than the watermark. The last windows were not written
by the stream (no newer event moved the watermark past their end) and are
only in the batch counts.

In [6]:
progress.groupBy("name").agg(
    F.count(F.lit(1)).alias("micro_batches"),
    F.sum("numInputRows").alias("input_rows"),
    F.round(F.avg("inputRowsPerSecond"), 1).alias("avg_input_rows_per_s"),
    F.round(F.avg("processedRowsPerSecond"), 1).alias("avg_processed_rows_per_s"),
    F.percentile("durationMs.triggerExecution", 0.5).alias("median_batch_ms"),
    F.max("durationMs.triggerExecution").alias("max_batch_ms"),
).toPandas()

,name,micro_batches,input_rows,avg_input_rows_per_s,avg_processed_rows_per_s,median_batch_ms,max_batch_ms
0,windows,52,3749,2.4,7.9,9328.0,20837
1,matches,52,3749,2.4,17.0,4496.0,13220


In [7]:
spark.stop()